# Robust Healthcare Decision Tree Assignment
This notebook dynamically cleans the uploaded dataset and avoids mixed-type encoder errors.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier,plot_tree
from sklearn.metrics import accuracy_score,classification_report,ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier

df=pd.read_csv(r""C:\Users\adipi\Downloads\synthetic_drug_data.csv"",engine="python",on_bad_lines="skip")
print(df.shape)
display(df.head())
print(df.columns.tolist())


In [ ]:
target='Seriousness'
df=df.dropna(subset=[target]).reset_index(drop=True)
drop=[c for c in ['ReportID'] if c in df.columns]
X=df.drop(columns=drop+[target]).copy()
y=df[target].astype(str)

for c in X.columns:
    X[c]=X[c].replace(['','nan','None'],np.nan)
    conv=pd.to_numeric(X[c],errors='coerce')
    if conv.notna().sum()/len(conv)>0.9:
        X[c]=conv.fillna(conv.median())
    else:
        X[c]=X[c].astype(str).fillna('Unknown')

# Force ALL non numeric to string
num_cols=[]
cat_cols=[]
for c in X.columns:
    if pd.api.types.is_numeric_dtype(X[c]):
        num_cols.append(c)
    else:
        X[c]=X[c].astype(str)
        cat_cols.append(c)

pre=ColumnTransformer([
('num',Pipeline([('imp',SimpleImputer(strategy='median'))]),num_cols),
('cat',Pipeline([
('imp',SimpleImputer(strategy='most_frequent')),
('enc',OneHotEncoder(handle_unknown='ignore',sparse_output=False))
]),cat_cols)
])

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


In [ ]:
pipe=Pipeline([
('pre',pre),
('clf',DecisionTreeClassifier(random_state=42))
])
pipe.fit(X_train,y_train)
pred=pipe.predict(X_test)
print("Accuracy:",accuracy_score(y_test,pred))
print(classification_report(y_test,pred))
ConfusionMatrixDisplay.from_predictions(y_test,pred)
plt.show()
print("Depth:",pipe.named_steps['clf'].get_depth())
print("Leaves:",pipe.named_steps['clf'].get_n_leaves())


In [ ]:
params={
'clf__max_depth':[5,10,None],
'clf__criterion':['gini','entropy']
}
grid=GridSearchCV(pipe,params,cv=3,n_jobs=-1)
grid.fit(X_train,y_train)
print(grid.best_params_)
print(grid.best_score_)
